# KGE Combination Extraction — LightOnOCR + GLiNER 2.0 (No Vocabularies)

Extract combinations `(model, dataset, metric)` from evaluation tables in KGE papers using **LightOnOCR-2-1B** for table extraction and **GLiNER 2.0** (`fastino/gliner2-base-v1`) for entity/relation recognition. No hardcoded vocabularies, normalization maps, or keyword lists.

**Pipeline:**
1. Setup (imports, paths)
2. Extract tables from PDFs with LightOnOCR
3. Load GLiNER 2.0 model
4. Detect entities row-by-row with GLiNER 2.0 → form combinations
5. Export to Excel

> Designed to run on a GPU server. On first run you may want to enable `AUTO_INSTALL_DEPS = True` to install `gliner2`, `transformers` (≥5), `pypdfium2`, `Pillow`, `accelerate` and `beautifulsoup4`.

## Section 1 — Setup: imports and paths

In [ ]:
# ─── Section 1 — Setup: imports and paths ─────────────────────────────────────
import json
import re
import warnings
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

SHOW_VERBOSE = False


def _find(candidates):
    return next((p for p in candidates if p.exists()), None)


PDF_DIR = _find([Path("pdfs_prueba"), Path("table_extraction/pdfs_prueba")])
assert PDF_DIR is not None, "Cannot locate pdfs_prueba/"

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))

print(f"PDF directory : {PDF_DIR}")
print(f"PDFs found    : {len(PDF_FILES)}")
for p in PDF_FILES:
    print(f"  • {p.name}")

## Section 2 — Table Extraction with LightOnOCR

Each PDF page is rendered at 200 DPI (longest side ≤ 1540 px — the value recommended by the LightOnOCR model card) and processed by `lightonai/LightOnOCR-2-1B`, which returns Markdown/HTML. We keep the `<table>…</table>` blocks.

Results are **cached** as `<stem>_lightonocr.json` inside `pdfs_prueba/` so subsequent runs are instant.

In [ ]:
# ─── Section 2a — Optional installation of OCR / GLiNER2 dependencies ────────
import sys, subprocess

AUTO_INSTALL_DEPS = False

INSTALL_SPECS = [
    "gliner2>=1.2.5",
    "transformers @ git+https://github.com/huggingface/transformers.git",
    "pypdfium2",
    "pillow",
    "accelerate",
    "beautifulsoup4",
    "openpyxl",
    # Needed for torch 2.11+cu130 to JIT-compile fused kernels (DeBERTa-v3 positional
    # buckets). Without this you get: "nvrtc: failed to open libnvrtc-builtins.so.13.0".
    "nvidia-cuda-nvrtc-cu13",
]

if AUTO_INSTALL_DEPS:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *INSTALL_SPECS]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    print("If transformers was updated you may need to restart the kernel.")

In [ ]:
# ─── Section 2b — Environment diagnostic ─────────────────────────────────────
import torch, transformers, sys as _sys

print("Python           :", _sys.version.split()[0])
print("torch            :", torch.__version__)
print("torch.cuda avail :", torch.cuda.is_available())
print("CUDA build       :", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
print("transformers     :", transformers.__version__)

try:
    from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor  # noqa: F401
    print("LightOnOcr       : OK")
except ImportError as e:
    print("LightOnOcr       : FAILED →", e)

try:
    import gliner2 as _gliner2
    print("gliner2          :", getattr(_gliner2, "__version__", "unknown"))
except ImportError as e:
    print("gliner2          : FAILED →", e)

In [ ]:
# ─── Section 2c — Load LightOnOCR model ──────────────────────────────────────
import pypdfium2 as pdfium
from PIL import Image
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

OCR_MODEL_ID = "lightonai/LightOnOCR-2-1B"
OCR_TARGET_LONGEST = 1540  # px, per LightOnOCR model card
OCR_MAX_NEW_TOKENS = 8192

if torch.cuda.is_available():
    ocr_device = "cuda"
    ocr_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    ocr_device = "mps"
    ocr_dtype = torch.float32
else:
    ocr_device = "cpu"
    ocr_dtype = torch.float32

print(f"OCR device: {ocr_device}, dtype: {ocr_dtype}")

ocr_processor = LightOnOcrProcessor.from_pretrained(OCR_MODEL_ID)
ocr_model = LightOnOcrForConditionalGeneration.from_pretrained(
    OCR_MODEL_ID,
    torch_dtype=ocr_dtype,
    attn_implementation="eager",
).to(ocr_device)

print(f"OCR model loaded : {OCR_MODEL_ID}")

In [ ]:
# ─── Section 2d — OCR helpers ────────────────────────────────────────────────
import tempfile, os


def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = OCR_TARGET_LONGEST) -> Image.Image:
    """Render a PDF page to PIL RGB image at 200 DPI, max `target_longest` px on the longest side."""
    page = pdf_doc[page_idx]
    bitmap = page.render(scale=200 / 72)  # 200 DPI
    pil_image = bitmap.to_pil()

    w, h = pil_image.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        pil_image = pil_image.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")
    return pil_image


def ocr_page(pil_image: Image.Image, max_new_tokens: int = OCR_MAX_NEW_TOKENS) -> str:
    """Run LightOnOCR on a page image. Returns the raw generated text (markdown/HTML)."""
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    pil_image.save(tmp, format="PNG")
    tmp.close()
    try:
        conversation = [
            {"role": "user", "content": [{"type": "image", "url": tmp.name}]}
        ]
        inputs = ocr_processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            k: v.to(device=ocr_device, dtype=ocr_dtype) if v.is_floating_point() else v.to(ocr_device)
            for k, v in inputs.items()
        }
        with torch.no_grad():
            output_ids = ocr_model.generate(**inputs, max_new_tokens=max_new_tokens)
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        return ocr_processor.decode(generated_ids, skip_special_tokens=True)
    finally:
        os.unlink(tmp.name)


def _extract_html_tables(text: str) -> list[str]:
    """Return every <table>…</table> block present in the OCR output."""
    return re.findall(r"<table\b[^>]*>.*?</table>", text, flags=re.DOTALL | re.IGNORECASE)


print("OCR helpers ready.")

In [ ]:
# ─── Section 2e — LightOnOCR extraction (with JSON cache) ────────────────────

def run_lightonocr(path_pdf: Path, cache_dir: Path, verbose: bool = False) -> dict:
    """Extract tables from one PDF using LightOnOCR. Cached as <stem>_lightonocr.json."""
    cache_file = cache_dir / f"{path_pdf.stem}_lightonocr.json"

    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            result = json.load(f)
        if verbose:
            n = sum(len(p["tables"]) for p in result["results"])
            print(f"[CACHE] {path_pdf.name} ({n} tables)")
        return result

    if verbose:
        print(f"[RUN] {path_pdf.name}")

    pdf_doc = pdfium.PdfDocument(str(path_pdf))
    results_data = []
    try:
        for page_idx in range(len(pdf_doc)):
            page_num = page_idx + 1
            pil_image = render_pdf_page(pdf_doc, page_idx)
            ocr_text = ocr_page(pil_image)
            html_tables = _extract_html_tables(ocr_text)
            if html_tables:
                results_data.append({
                    "page": page_num,
                    "tables": [{"html": t} for t in html_tables],
                })
    finally:
        pdf_doc.close()

    result = {"file_name": str(path_pdf), "results": results_data}
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result


ocr_outputs: dict = {}
extraction_rows = []

# Build the list of papers to process. Prefer PDFs (source of truth) but fall
# back to existing *_lightonocr.json caches if the raw PDFs are missing — this
# keeps the downstream GLiNER2 pipeline usable even after someone accidentally
# deletes the PDFs.
cache_files = sorted(PDF_DIR.glob("*_lightonocr.json"))
to_process: list[tuple[str, Path | None]] = []

if PDF_FILES:
    to_process = [(pdf.stem, pdf) for pdf in PDF_FILES]
elif cache_files:
    print(
        f"⚠  No PDFs found in {PDF_DIR.resolve()}, but {len(cache_files)} "
        f"*_lightonocr.json caches are present. Using caches directly "
        f"(LightOnOCR cannot be re-run without the source PDFs)."
    )
    to_process = [
        (cache.name.replace("_lightonocr.json", ""), None)
        for cache in cache_files
    ]
else:
    raise FileNotFoundError(
        f"No PDFs and no *_lightonocr.json caches found in "
        f"{PDF_DIR.resolve()}. Restore either the PDFs or the JSON caches. "
        f"(CWD = {Path.cwd()})."
    )

for stem, pdf in to_process:
    cache_file = PDF_DIR / f"{stem}_lightonocr.json"
    if pdf is not None:
        source = "cache" if cache_file.exists() else "run"
        ocr_outputs[stem] = run_lightonocr(pdf, cache_dir=PDF_DIR, verbose=SHOW_VERBOSE)
    else:
        source = "cache-only"
        with open(cache_file, "r", encoding="utf-8") as f:
            ocr_outputs[stem] = json.load(f)
    n_tables = sum(len(p["tables"]) for p in ocr_outputs[stem]["results"])
    extraction_rows.append({"paper": stem, "source": source, "tables": n_tables})

extract_df = pd.DataFrame(
    extraction_rows,
    columns=["paper", "source", "tables"],
).sort_values(["tables", "paper"], ascending=[False, True])
print(f"\nExtracted tables from {len(to_process)} papers (total: {int(extract_df['tables'].sum())})")
display(extract_df)

In [ ]:
# ─── Section 3a — Disable PyTorch JIT / TensorExpr fusion ────────────────────
# DeBERTa-v3 (the encoder used by gliner2) triggers a TensorExpr fusion on the
# relative-position-bucket path, which calls nvrtc at runtime. On some servers
# (e.g. torch 2.11+cu130 without `nvidia-cuda-nvrtc-cu13`) this fails with:
#   RuntimeError: nvrtc: error: failed to open libnvrtc-builtins.so.13.0
# Disabling the fusers forces eager kernels (tiny slowdown, fully correct).
import os
os.environ.setdefault("PYTORCH_JIT", "0")
os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")

import torch
for name, args in [
    ("_jit_set_profiling_executor", (False,)),
    ("_jit_set_profiling_mode",     (False,)),
    ("_jit_override_can_fuse_on_gpu", (False,)),
    ("_jit_override_can_fuse_on_cpu", (False,)),
    ("_jit_set_texpr_fuser_enabled", (False,)),
    ("_jit_set_nvfuser_enabled",     (False,)),
]:
    fn = getattr(torch._C, name, None)
    if fn is not None:
        try:
            fn(*args)
        except Exception as _e:
            print(f"  (skipped torch._C.{name}: {_e})")

print("JIT / TensorExpr / nvFuser fusers disabled.")


## Section 3 — GLiNER 2.0 Dependency Check

Load GLiNER 2.0 model dependencies for combination extraction.

In [ ]:
# ─── Section 3 — GLiNER 2.0 model ────────────────────────────────────────────
from gliner2 import GLiNER2
import gliner2 as _gliner2

GLINER2_MODEL_ID = "fastino/gliner2-base-v1"   # base has better recall on KGE tables than large

gliner2_map_location = "cuda" if torch.cuda.is_available() else "cpu"

gliner2_model = GLiNER2.from_pretrained(
    GLINER2_MODEL_ID,
    map_location=gliner2_map_location,
)

print(f"GLiNER2 available: {getattr(_gliner2, '__version__', 'unknown')}")
print(f"GLiNER2 model    : {GLINER2_MODEL_ID} on {gliner2_map_location}")

In [ ]:
# ─── Section 3b — GLiNER 2.0 smoke test ──────────────────────────────────────
# Run a quick sanity check on two representative inputs so we can confirm the model
# and our schema actually produce something before crunching all 17 tables.
_DEMO_DESCRIPTIONS = {
    "model": "Name of a machine-learning model, knowledge-graph embedding method, algorithm or system (e.g. TransE, ComplEx, HolE, RotatE).",
    "dataset": "Name of a benchmark dataset or knowledge-graph corpus (e.g. WN18, WN18RR, FB15k, FB15k-237, YAGO, NELL).",
    "metric": "Name of an evaluation metric used to score a model (e.g. MRR, Hits@1, Hits@3, Hits@10, MR, Accuracy, F1).",
}

_DEMO_TEXTS = [
    # (A) prose that clearly contains all three entity types
    "We compare TransE and ComplEx on WN18 and FB15k-237 using MRR and Hits@10.",
    # (B) a raw row as produced by _rows_from_html on a LightOnOCR table
    "Models | WN18 | | | | FB15k | | |",
    "TransE* | 45.4 | 8.9 | 82.3 | 93.4 | 38.0 | 23.1 | 47.2 | 64.1",
]

print("── GLiNER 2.0 smoke test ──")
for txt in _DEMO_TEXTS:
    print(f"\nINPUT : {txt}")
    # Plain call (what the extraction pipeline uses)
    try:
        r_plain = gliner2_model.extract_entities(txt, _DEMO_DESCRIPTIONS)
        print(f"plain : {r_plain}")
    except Exception as e:
        print(f"plain : ERROR {type(e).__name__}: {e}")
    # With confidence (to see scores and decide a threshold)
    try:
        r_conf = gliner2_model.extract_entities(txt, _DEMO_DESCRIPTIONS, include_confidence=True)
        print(f"conf  : {r_conf}")
    except Exception as e:
        print(f"conf  : ERROR {type(e).__name__}: {e}")


## Section 4 — Pure GLiNER 2.0 combination extraction (no vocabularies)

In [ ]:
# ─── Section 4 — Pure GLiNER 2.0 combination extraction ──────────────────────────

GLINER2_LABEL_DESCRIPTIONS = {
    "model": "Name of a machine-learning model, knowledge-graph embedding method, algorithm or system (e.g. TransE, ComplEx, HolE, RotatE).",
    "dataset": "Name of a benchmark dataset or knowledge-graph corpus (e.g. WN18, WN18RR, FB15k, FB15k-237, YAGO, NELL).",
    "metric": "Name of an evaluation metric used to score a model (e.g. MRR, Hits@1, Hits@3, Hits@10, MR, Accuracy, F1).",
}
GLINER2_LABELS = list(GLINER2_LABEL_DESCRIPTIONS.keys())
GLINER2_MIN_SCORE = 0.5            # raise threshold to cut noisy cross-labeled entities
GLINER2_MAX_CHARS = 3000
GLINER2_USE_CONFIDENCE = True      # verified working after the JIT fix
GLINER2_DEBUG_ERRORS = True        # print the first N extraction errors instead of swallowing them
_gliner2_error_count = {"n": 0}
_GLINER2_ERROR_LIMIT = 3

# Toggle for the domain post-filter. Flip to False to see the raw GLiNER 2.0
# output with no corpus-level heuristics (useful for an honest comparison in
# ablations and for validating that the blacklist is not masking real entities).
USE_BLACKLIST: bool = True

# Known false positives for this KGE corpus: terms that gliner2-base occasionally
# mislabels. Keys are the target label, values are (lowercased) texts to drop.
# Kept intentionally conservative — only add items that are NEVER a valid instance
# of that label in the KGE literature.
GLINER2_LABEL_BLACKLIST: dict[str, set[str]] = {
    "model": {
        "model", "models", "method", "our", "only",  # literal column headers / stopwords
        "adagrad",                                   # optimizer, not a KGE model
        "bern",                                      # negative-sampling strategy
        "learning rate", "model size", "embedding size",
        "wd",                                         # WordNet/Wikidata dataset
        "translational",                              # category adjective, not a model
        "negs_e", "negs_r",                           # negative-sampling configurations
    },
    "dataset": {
        "lmf", "sme",                                # KGE models, not datasets
        "bern",                                      # neg-sampling strategy
        "gru",                                       # recurrent unit, not a dataset
        "n_e", "n_r", "|e|", "|r|",                  # entity/relation count symbols
        "filter", "raw", "baseline",                 # ranking-protocol modifiers
        "berlin", "germany",                         # spurious city/country names
        "ent", "iw", "sc", "rw",                     # metric abbreviations, not datasets
    },
    "metric": {
        "hyperkg",                                   # self-reference: model name leaking
        "beta",                                      # hyperparameter, not a metric
        "filter", "raw", "baseline",                 # ranking-protocol modifiers
        "only", "method", "models", "model",         # column-header leaks
        "learning rate", "model size",               # hyperparameters
        "wd",                                         # dataset name, not a metric
        "n_e", "n_r",                                 # entity/relation counts
        "nce baseline", "nce", "rw",                  # non-metric modifiers
    },
}


# ── Helpers ──────────────────────────────────────────────────────────────────

def _rows_from_html(html: str) -> list[str]:
    """One text string per <tr>, cells joined by ' | '."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.find_all("tr"):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        if cells:
            rows.append(" | ".join(cells))
    return rows


def _header_and_rows_from_html(html: str) -> tuple[list[str], list[str]]:
    """Split an HTML table into (header_lines, body_lines).

    A row is treated as header if it lives inside <thead> OR if every cell is
    a <th>. If no header is detected, the first row is used as header (common
    in OCR output that omits <thead>).
    """
    soup = BeautifulSoup(html, "html.parser")
    thead = soup.find("thead")
    tbody = soup.find("tbody")

    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None

    header_lines: list[str] = []
    body_lines: list[str] = []

    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)

    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for i, tr in enumerate(trs):
        # skip rows already consumed by thead
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        all_th = all(c.name == "th" for c in cells)
        if all_th and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)

    # Fallback: if we still have no header but do have body rows, use the first
    # body row as the header (common in LightOnOCR output without <thead>).
    if not header_lines and body_lines:
        header_lines = [body_lines[0]]
        body_lines = body_lines[1:]

    return header_lines, body_lines


def _clean_entity(value: str) -> str:
    """Light cleanup: strip citation markers, LaTeX math mode and whitespace."""
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)                                   # [1], [Smith 2020]
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    # LaTeX math mode: $...$  →  content inside
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    # Common LaTeX wrappers (must run BEFORE stripping braces)
    s = re.sub(r"\\(?:textbf|textit|text|mathbf|mathrm|mathit)\{([^{}]*)\}", r"\1", s)
    # Sub/super-scripts: _{xxx}, ^{xxx}  →  xxx
    for _ in range(3):
        s = re.sub(r"[_^]\{([^{}]*)\}", r"\1", s)
    # Remaining stray braces
    s = s.replace("{", "").replace("}", "")
    # Tighten "WD ++" / "WD --" that came from "WD $_{++}$" etc.
    s = re.sub(r"(\w)\s+(\+\+|--)(?=\s|$)", r"\1\2", s)
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    return s


def _extract_entities_raw(text: str) -> dict[str, tuple[str, float]]:
    """Return the raw best-label-per-text mapping (pre-dedup to the label dict).

    Returned shape: {entity_text: (label, confidence)}. The caller can then
    aggregate across rows of the same table before assigning final labels.
    """
    if not text.strip():
        return {}
    try:
        if GLINER2_USE_CONFIDENCE:
            result = gliner2_model.extract_entities(
                text[:GLINER2_MAX_CHARS],
                GLINER2_LABEL_DESCRIPTIONS,
                include_confidence=True,
            )
        else:
            result = gliner2_model.extract_entities(
                text[:GLINER2_MAX_CHARS],
                GLINER2_LABEL_DESCRIPTIONS,
            )
    except Exception as e:
        if GLINER2_DEBUG_ERRORS and _gliner2_error_count["n"] < _GLINER2_ERROR_LIMIT:
            _gliner2_error_count["n"] += 1
            print(f"[gliner2 error #{_gliner2_error_count['n']}] {type(e).__name__}: {e}")
            print(f"  on text: {text[:200]!r}")
        return {}

    ents_by_label = (result or {}).get("entities", {}) or {}
    best: dict[str, tuple[str, float]] = {}
    for label, items in ents_by_label.items():
        if label not in GLINER2_LABELS:
            continue
        for item in items or []:
            if isinstance(item, dict):
                raw_text = str(item.get("text", ""))
                score = float(item.get("confidence", 1.0) or 1.0)
            else:
                raw_text = str(item)
                score = 1.0
            value = _clean_entity(raw_text)
            if score < GLINER2_MIN_SCORE or not value or len(value) < 2:
                continue
            if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
                continue
            if USE_BLACKLIST and value.lower() in GLINER2_LABEL_BLACKLIST.get(label, set()):
                continue
            prev = best.get(value)
            if prev is None or score > prev[1]:
                best[value] = (label, score)
    return best


def _extract_entities(text: str) -> dict[str, list[str]]:
    """Run GLiNER 2.0 on a text fragment and return detected entities per label
    with per-call dedup (each entity text lands under its highest-confidence label)."""
    best = _extract_entities_raw(text)
    found: dict[str, set[str]] = {label: set() for label in GLINER2_LABELS}
    for value, (label, _score) in best.items():
        found[label].add(value)
    return {k: sorted(v) for k, v in found.items()}


def _merge_best(target: dict[str, tuple[str, float]], other: dict[str, tuple[str, float]]) -> None:
    """Merge `other` into `target`, keeping the highest-confidence label per text."""
    for value, (label, score) in other.items():
        prev = target.get(value)
        if prev is None or score > prev[1]:
            target[value] = (label, score)


def _labels_from_best(best: dict[str, tuple[str, float]]) -> dict[str, list[str]]:
    found: dict[str, set[str]] = {label: set() for label in GLINER2_LABELS}
    for value, (label, _score) in best.items():
        found[label].add(value)
    return {k: sorted(v) for k, v in found.items()}


def _combinations_from_entities(ents: dict[str, list[str]]) -> list[tuple[str, str, str]]:
    return [(m, d, mt) for m in ents["model"] for d in ents["dataset"] for mt in ents["metric"]]


# ── Main extraction loop ────────────────────────────────────────────────────

combination_rows: list[dict] = []
table_meta_rows: list[dict] = []

for pdf_stem, ocr_result in ocr_outputs.items():
    for page_data in ocr_result.get("results", []):
        page_num = int(page_data.get("page", 0) or 0)
        for table_idx, table in enumerate(page_data.get("tables", []), start=1):
            table_name = f"{pdf_stem}_p{page_num}_t{table_idx}"
            html = table.get("html", "")
            if not html:
                continue

            header_lines, body_lines = _header_and_rows_from_html(html)
            if not header_lines and not body_lines:
                continue

            header_context = "\n".join(header_lines)

            # --- Pass 1: header alone (anchors dataset/metric labels strongly) ---
            table_best: dict[str, tuple[str, float]] = {}
            if header_context:
                _merge_best(table_best, _extract_entities_raw(header_context))

            # --- Pass 2: each body row, prefixed with the header for context ---
            for row_text in body_lines:
                if header_context:
                    prompt = (
                        f"Table column headers: {header_context}\n"
                        f"Row: {row_text}"
                    )
                else:
                    prompt = row_text
                _merge_best(table_best, _extract_entities_raw(prompt))

            # --- Fallback: feed the whole table as a single chunk if nothing found
            if not table_best:
                full = "\n".join(header_lines + body_lines)
                _merge_best(table_best, _extract_entities_raw(full))

            # After both passes, resolve each unique text to a single label (the
            # one with highest confidence across all detections). This prevents
            # "LMF" from leaking into `dataset` when it's really a `model`.
            ents = _labels_from_best(table_best)
            all_models = set(ents["model"])
            all_datasets = set(ents["dataset"])
            all_metrics = set(ents["metric"])

            # --- Form combinations: model × dataset × metric at table level ---
            table_combinations: set[tuple[str, str, str]] = set(_combinations_from_entities(ents))

            table_meta_rows.append({
                "paper": pdf_stem,
                "table_name": table_name,
                "models": " | ".join(sorted(all_models)) or "(none)",
                "datasets": " | ".join(sorted(all_datasets)) or "(none)",
                "metrics": " | ".join(sorted(all_metrics)) or "(none)",
                "combinations_found": len(table_combinations),
            })

            for m, d, mt in sorted(table_combinations):
                combination_rows.append({
                    "paper": pdf_stem,
                    "table_name": table_name,
                    "model": m,
                    "dataset": d,
                    "metric": mt,
                })

# ── Build dataframes ────────────────────────────────────────────────────────

combinations_df = pd.DataFrame(combination_rows) if combination_rows else pd.DataFrame(
    columns=["paper", "table_name", "model", "dataset", "metric"]
)
combinations_df = combinations_df.drop_duplicates().sort_values(
    ["paper", "table_name", "model", "dataset", "metric"]
).reset_index(drop=True)

table_meta_df = pd.DataFrame(table_meta_rows)

print("=" * 70)
print("PURE GLINER 2.0 COMBINATION EXTRACTION (LightOnOCR tables)")
print("=" * 70)
print(f"Blacklist mode : {'ON (post-filter applied)' if USE_BLACKLIST else 'OFF (raw GLiNER2 output)'}")
print(f"Total combinations : {len(combinations_df)}")
print(f"Unique models  : {combinations_df['model'].nunique() if len(combinations_df) else 0}")
print(f"Unique datasets: {combinations_df['dataset'].nunique() if len(combinations_df) else 0}")
print(f"Unique metrics : {combinations_df['metric'].nunique() if len(combinations_df) else 0}")
print()
display(combinations_df)
print()
print("Table metadata:")
display(table_meta_df)

In [ ]:
# ─── Tables with metrics: full detail (combinations if available, partial otherwise)

metric_table_rows = []

# Tables that have at least one metric detected
tables_with_metrics = table_meta_df[table_meta_df["metrics"] != "(none)"].copy()

for _, meta in tables_with_metrics.iterrows():
    paper = meta["paper"]
    tname = meta["table_name"]

    # Get combinations for this table (if any)
    table_trips = combinations_df[
        (combinations_df["paper"] == paper) & (combinations_df["table_name"] == tname)
    ]

    if not table_trips.empty:
        # Table has full combinations -> one row per combination
        for _, trip in table_trips.iterrows():
            metric_table_rows.append({
                "paper": paper,
                "table_name": tname,
                "model": trip["model"],
                "dataset": trip["dataset"],
                "metric": trip["metric"],
                "has_combination": True,
            })
    else:
        # Table has metrics but no complete combination -> include partial info
        models = meta["models"] if meta["models"] != "(none)" else ""
        datasets = meta["datasets"] if meta["datasets"] != "(none)" else ""
        metrics_list = [m.strip() for m in meta["metrics"].split("|")]
        model_list = [m.strip() for m in models.split("|")] if models else [""]
        dataset_list = [d.strip() for d in datasets.split("|")] if datasets else [""]

        for metric in metrics_list:
            for model in model_list:
                for dataset in dataset_list:
                    metric_table_rows.append({
                        "paper": paper,
                        "table_name": tname,
                        "model": model,
                        "dataset": dataset,
                        "metric": metric,
                        "has_combination": False,
                    })

tables_with_metrics_df = pd.DataFrame(metric_table_rows)
if not tables_with_metrics_df.empty:
    tables_with_metrics_df = tables_with_metrics_df.drop_duplicates().sort_values(
        ["paper", "table_name", "model", "dataset", "metric"]
    ).reset_index(drop=True)

n_with = int(tables_with_metrics_df["has_combination"].sum()) if len(tables_with_metrics_df) else 0
n_without = len(tables_with_metrics_df) - n_with

print(f"Tables with metrics: {len(tables_with_metrics)}")
print(f"  Rows with full combination    : {n_with}")
print(f"  Rows partial (no combination) : {n_without}")
print()
display(tables_with_metrics_df)

In [ ]:
# ─── Tables With Values: same detail as "Tables With Metrics" + numeric value
# Looks up, for every (paper, table_name, model, dataset, metric) row, the cell
# value at the intersection of the model row and the column whose header text
# matches the dataset + metric. The output mirrors `tables_with_metrics_df`
# but adds a `value` column.

import re

_NUMERIC_RE = re.compile(r"[-+]?\d+(?:[.,]\d+)?")


def _parse_table_grid(
    html: str,
) -> tuple[list[list[str]], list[bool], list[list[bool]], int]:
    """Parse HTML table into a 2D text grid, expanding colspan/rowspan.

    Returns (grid, is_header_mask, fixed_mask, max_cols) where:
      * `grid[r][c]` is the cell text after expansion (rows may be ragged —
        padding is left to the caller / rebalance step).
      * `is_header_mask[r]` is True iff every cell in row r is a <th>.
      * `fixed_mask[r][c]` is True iff the cell was declared with rowspan>1
        or was inherited from a rowspan in a previous row. These cells must
        NOT be shifted by the rebalance step.
      * `max_cols` is the widest row (usually the body / leaf-header width).
    """
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table") or soup
    tr_list = table.find_all("tr")

    grid: list[list[str]] = []
    is_header: list[bool] = []
    fixed: list[list[bool]] = []
    pending: dict[int, tuple[str, int]] = {}  # col_idx -> (text, rows_left)

    def _ensure(row, fmask, n):
        while len(row) <= n:
            row.append("")
            fmask.append(False)

    for tr in tr_list:
        row: list[str] = []
        fmask: list[bool] = []
        cells = tr.find_all(["td", "th"])
        is_header.append(bool(cells) and all(c.name == "th" for c in cells))

        c_idx = 0
        for cell in cells:
            while pending.get(c_idx, (None, 0))[1] > 0:
                text, rem = pending[c_idx]
                _ensure(row, fmask, c_idx)
                row[c_idx] = text
                fmask[c_idx] = True  # inherited from rowspan
                pending[c_idx] = (text, rem - 1)
                c_idx += 1

            text = re.sub(r"\s+", " ", cell.get_text(separator=" ", strip=True))
            colspan = int(cell.get("colspan", 1) or 1)
            rowspan = int(cell.get("rowspan", 1) or 1)
            is_fixed = rowspan > 1
            for _ in range(colspan):
                _ensure(row, fmask, c_idx)
                row[c_idx] = text
                fmask[c_idx] = is_fixed
                if is_fixed:
                    pending[c_idx] = (text, rowspan - 1)
                c_idx += 1

        for col in sorted(pending):
            if pending[col][1] > 0 and col >= len(row):
                text, rem = pending[col]
                while len(row) < col:
                    row.append("")
                    fmask.append(False)
                row.append(text)
                fmask.append(True)
                pending[col] = (text, rem - 1)

        grid.append(row)
        fixed.append(fmask)

    max_cols = max((len(r) for r in grid), default=0)
    return grid, is_header, fixed, max_cols


def _rebalance_header_grid(
    grid: list[list[str]],
    is_header: list[bool],
    fixed: list[list[bool]],
    max_cols: int,
) -> list[list[str]]:
    """Fix header rows that are narrower than the body, redistributing
    non-fixed (non-rowspan) cells evenly across the body width.

    This is needed because LightOnOCR sometimes emits outer header cells with
    colspan that doesn't match the leaf-row width (e.g. `<th colspan="2">WN18`
    when the leaf row actually has 4 sub-columns under WN18). Without this
    fix, column signatures would be misaligned and (dataset, metric) lookups
    would return the wrong values.

    Fixed cells (rowspan > 1, or inherited from a rowspan) keep their
    original column index. Non-fixed cells are spread uniformly across the
    remaining columns. Rows whose width already matches `max_cols`, or where
    the mismatch cannot be resolved with an integer factor, are just padded.
    """
    new_grid: list[list[str]] = []
    for r_idx, row in enumerate(grid):
        cur_w = len(row)
        if not is_header[r_idx] or cur_w == max_cols:
            padded = list(row) + [""] * (max_cols - cur_w)
            new_grid.append(padded)
            continue

        fmask = fixed[r_idx]
        explicit_idx = [i for i, f in enumerate(fmask) if not f]
        fixed_idx = [i for i, f in enumerate(fmask) if f]
        n_exp = len(explicit_idx)
        n_fix = len(fixed_idx)

        if n_exp == 0:
            new_grid.append(list(row) + [""] * (max_cols - cur_w))
            continue

        target = max_cols - n_fix
        factor, rem = divmod(target, n_exp)
        if factor <= 1 or rem != 0:
            # Can't rebalance cleanly — fall back to padding
            new_grid.append(list(row) + [""] * (max_cols - cur_w))
            continue

        new_row = [""] * max_cols
        # Place fixed cells at their original column indices
        fixed_cols_set = set()
        for i in fixed_idx:
            if i < max_cols:
                new_row[i] = row[i]
                fixed_cols_set.add(i)
        # Remaining column slots, in order, receive explicit cells (each
        # cell expanded to `factor` contiguous columns).
        free = [c for c in range(max_cols) if c not in fixed_cols_set]
        if len(free) != target:
            new_grid.append(list(row) + [""] * (max_cols - cur_w))
            continue
        pos = 0
        for i in explicit_idx:
            for _ in range(factor):
                new_row[free[pos]] = row[i]
                pos += 1
        new_grid.append(new_row)

    return new_grid


def _looks_numeric(text: str) -> bool:
    if not text:
        return False
    s = text.replace(",", "").replace("%", "").replace("$", "")
    s = s.replace("\\", "").replace("_", "").replace("^", "").replace("±", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return bool(s) and any(_NUMERIC_RE.fullmatch(tok) for tok in s.split())


def _norm_for_match(s: str) -> str:
    s = (s or "").lower().replace("(", " ").replace(")", " ").replace("%", " ")
    return re.sub(r"\s+", " ", s).strip()


def _values_from_table(
    html: str, model: str, dataset: str, metric: str,
) -> list[tuple[str, str]]:
    """Return all (variant, value) pairs at (row=model, col≈dataset+metric).

    When a (model, dataset, metric) cell is subdivided by a third header level
    (e.g. 'raw' / 'filter' under each metric), each sub-column becomes one
    (variant, value) pair where `variant` is the leaf header text of that
    column. If there is no sub-variant (leaf equals the metric or dataset
    itself), `variant` is "". Matching is case-insensitive and tolerant of
    whitespace, parentheses and percent signs so 'Hits@10(%)' cuadra con
    'Hits@10 (%)' y 'FB15K' con 'FB15k'. Returns [] when nothing matches.
    """
    if not html or not (model or metric):
        return []
    try:
        grid, is_header, fixed, max_cols = _parse_table_grid(html)
        grid = _rebalance_header_grid(grid, is_header, fixed, max_cols)
    except Exception:
        return []
    if not grid or not any(is_header):
        return []

    last_header = max(i for i, h in enumerate(is_header) if h)
    n_cols = max_cols

    # For each column, build the full header path (list of distinct texts top→bottom)
    col_path: list[list[str]] = []
    for c in range(n_cols):
        path: list[str] = []
        for r in range(last_header + 1):
            cell = grid[r][c] if c < len(grid[r]) else ""
            if cell and (not path or path[-1] != cell):
                path.append(cell)
        col_path.append(path)
    col_sig = [_norm_for_match(" ".join(p)) for p in col_path]
    col_leaf = [p[-1] if p else "" for p in col_path]

    m_norm = _norm_for_match(metric)
    d_norm = _norm_for_match(dataset)

    # Prefer columns whose signature contains BOTH dataset and metric
    cand_cols = [
        c for c, key in enumerate(col_sig)
        if (not d_norm or d_norm in key) and (not m_norm or m_norm in key)
    ]
    # Fallback: match metric only (tables with a single dataset)
    if not cand_cols and m_norm:
        cand_cols = [c for c, key in enumerate(col_sig) if m_norm in key]
    if not cand_cols:
        return []

    def _variant_for(col_idx: int) -> str:
        """Return the leaf header of this column, or '' if it just echoes
        the metric / dataset (i.e. the column has no sub-variant)."""
        leaf = col_leaf[col_idx].strip()
        leaf_norm = _norm_for_match(leaf)
        if not leaf_norm:
            return ""
        if m_norm and (leaf_norm == m_norm or leaf_norm in m_norm or m_norm in leaf_norm):
            return ""
        if d_norm and (leaf_norm == d_norm or leaf_norm in d_norm or d_norm in leaf_norm):
            return ""
        return leaf

    model_norm = _norm_for_match(model)
    for r in range(last_header + 1, len(grid)):
        row = grid[r]
        if not any(_looks_numeric(x) for x in row):
            continue
        # Leading non-numeric cells form the row label
        leading: list[str] = []
        for cell in row:
            if _looks_numeric(cell):
                break
            leading.append(cell)
        lead_norm = _norm_for_match(" ".join(leading))
        if not (model_norm and model_norm in lead_norm):
            continue
        # Collect one (variant, value) per candidate column in this row
        results: list[tuple[str, str]] = []
        seen: set[tuple[str, str]] = set()
        # Prefer numeric cells; fall back to any non-empty cell
        for c in cand_cols:
            if c >= len(row):
                continue
            val = row[c].strip()
            if not val or not _looks_numeric(val):
                continue
            variant = _variant_for(c)
            key = (variant, val)
            if key in seen:
                continue
            seen.add(key)
            results.append((variant, val))
        if not results:
            for c in cand_cols:
                if c < len(row) and row[c].strip():
                    variant = _variant_for(c)
                    key = (variant, row[c].strip())
                    if key in seen:
                        continue
                    seen.add(key)
                    results.append((variant, row[c].strip()))
        if results:
            return results
    return []


# Index: table_name -> raw HTML (once)
_html_by_table: dict[str, str] = {}
for _stem, _res in ocr_outputs.items():
    for _pd in _res.get("results", []):
        _pn = int(_pd.get("page", 0) or 0)
        for _ti, _tb in enumerate(_pd.get("tables", []), start=1):
            _html_by_table[f"{_stem}_p{_pn}_t{_ti}"] = _tb.get("html", "")


_tw_values_rows: list[dict] = []
for _, r in tables_with_metrics_df.iterrows():
    html = _html_by_table.get(r["table_name"], "")
    pairs = _values_from_table(html, r["model"], r["dataset"], r["metric"])
    if not pairs:
        # Keep one row so the model/dataset/metric is still represented,
        # even if we couldn't resolve a numeric value.
        _tw_values_rows.append({
            "paper": r["paper"],
            "table_name": r["table_name"],
            "model": r["model"],
            "dataset": r["dataset"],
            "metric": r["metric"],
            "variant": "",
            "value": "",
            "has_combination": bool(r["has_combination"]),
        })
        continue
    for variant, val in pairs:
        _tw_values_rows.append({
            "paper": r["paper"],
            "table_name": r["table_name"],
            "model": r["model"],
            "dataset": r["dataset"],
            "metric": r["metric"],
            "variant": variant,
            "value": val,
            "has_combination": bool(r["has_combination"]),
        })

tables_with_values_df = pd.DataFrame(_tw_values_rows)

_n_val = int((tables_with_values_df["value"] != "").sum())
_n_variant = int((tables_with_values_df["variant"] != "").sum())
print(f"Tables With Values: {len(tables_with_values_df)} rows")
print(f"  Rows with numeric value : {_n_val}")
print(f"  Rows without value      : {len(tables_with_values_df) - _n_val}")
print(f"  Rows with sub-variant   : {_n_variant}  (e.g. 'raw' / 'filter')")
display(tables_with_values_df.head(25))

## Section 5 — Export to Excel

In [ ]:
# ─── Section 5 — Export to Excel ─────────────────────────────────────────────
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

_mode_tag = "filtered" if USE_BLACKLIST else "raw"
export_file = PDF_DIR / f"gliner2_lightonocr_combinations_{_mode_tag}.xlsx"

with pd.ExcelWriter(export_file, engine="openpyxl") as writer:
    combinations_df.to_excel(writer, index=False, sheet_name="Combinations")
    table_meta_df.to_excel(writer, index=False, sheet_name="Table Metadata")
    tables_with_metrics_df.to_excel(writer, index=False, sheet_name="Tables With Metrics")
    tables_with_values_df.to_excel(writer, index=False, sheet_name="Tables With Values")

    wb = writer.book
    header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
    header_font = Font(bold=True)

    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
        for col_idx, col_cells in enumerate(
            ws.iter_cols(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1
        ):
            max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(10, max_len + 2), 60)

print(f"Excel: {export_file}")
print(f"  Combinations sheet         : {len(combinations_df)} rows")
print(f"  Metadata sheet             : {len(table_meta_df)} rows")
print(f"  Tables With Metrics sheet  : {len(tables_with_metrics_df)} rows")
print(f"  Tables With Values sheet   : {len(tables_with_values_df)} rows")